# 03_chunking_strategies: Parent-Child Hierarchical RAG

This notebook splits scraped Wikipedia paragraphs into large parent and small child chunks, indexing them in a relational FAISS layout.


In [1]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

load_dotenv(dotenv_path=r"d:\Study\Prep\.env")

# 1. Scrape document
url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.content, "html.parser")
parent_doc = "\n".join([p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 100][:4])

# 2. Setup splitters
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=0)

parents = parent_splitter.split_text(parent_doc)
child_texts = []
child_metadatas = []

# Link child chunks to parent contexts in metadata
for idx, p_text in enumerate(parents):
    children = child_splitter.split_text(p_text)
    for c_text in children:
        child_texts.append(c_text)
        child_metadatas.append({"parent_content": p_text, "parent_idx": idx})

# 3. Index children
embeddings = OpenAIEmbeddings()
db = FAISS.from_texts(child_texts, embeddings, metadatas=child_metadatas)

# 4. Retrieve
query = "Facebook AI Research Lewis"
retrieved = db.similarity_search(query, k=1)[0]

print("Retrieved Child Chunk:", retrieved.page_content[:100] + "...")
print("Resolved Parent Context:", retrieved.metadata["parent_content"][:200] + "...")


C:\Users\aryan\AppData\Local\Temp\ipykernel_26324\3875384809.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Retrieved Child Chunk: and has since become a widely adopted approach in modern AI systems....
Resolved Parent Context: information that is not available in the training data.[2] For example, this enables LLM-based chatbots to access internal company data or generate responses based on authoritative sources. The techni...


### Output Explanation
- Child vectors optimize matching precision.
- The system resolves the parent context metadata tag (`parent_content`), returning the full parent text block to prevent context truncation.
